# Lecture 5: Ridge, RidgeCV and ElasticNet

## Algerian Forest Fires project

Goal: predict the **FWI (Fire Weather Index)**. This lesson continues the previous notebook by comparing three regularized regression choices.

**Road map:** prepare data → Ridge → RidgeCV → ElasticNet → ElasticNetCV → choose a model using the final test result.

## The idea in simple English

- **Ridge** adds an L2 penalty: alpha × sum of coefficient squares. It shrinks coefficients but usually does not make them exactly zero.
- **Lasso** uses an L1 penalty: alpha × sum of absolute coefficients. It can remove features by making coefficients zero.
- **ElasticNet** combines L1 and L2. It can shrink, select, and cope well with related features.

**Important correction:** the absolute-value penalty belongs to Lasso, not Ridge.

In [ ]:
# Cell 1: imports and data loading
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, Ridge, RidgeCV, ElasticNet, ElasticNetCV
from sklearn.metrics import mean_absolute_error, r2_score

sns.set_theme(style='whitegrid')
df = pd.read_csv('Model Training Practicals/Algerian_forest_fires_cleaned_dataset.csv')
df.head()

### What changed here?

We import the four new estimators. The CSV is the cleaned project data used in the earlier training notebook. Keeping one fixed dataset makes the model comparison fair.

In [ ]:
# Cell 2: create input features and the FWI target
df['Classes'] = (df['Classes'].astype(str).str.strip().str.lower()
                 .map({'not fire': 0, 'fire': 1}))
target = 'FWI'
X = df.drop(columns=['day', 'month', 'year', target])
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
print('Training shape:', X_train.shape)
print('Test shape:', X_test.shape)

### Why the pipeline matters

Ridge and ElasticNet are sensitive to feature scales. A pipeline first standardizes the data and then fits the model. During cross-validation, each fold learns scaling values from its own training portion only. This prevents information leaking from validation or test data.

In [ ]:
# Cell 3: visualise how Ridge changes coefficient size
alphas_for_plot = np.logspace(-3, 3, 80)
coefficient_paths = []
for alpha in alphas_for_plot:
    ridge = make_pipeline(StandardScaler(), Ridge(alpha=alpha))
    ridge.fit(X_train, y_train)
    coefficient_paths.append(ridge.named_steps['ridge'].coef_)

plt.figure(figsize=(9, 5))
plt.semilogx(alphas_for_plot, coefficient_paths)
plt.xlabel('Alpha: regularization strength')
plt.ylabel('Coefficient value')
plt.title('Ridge shrinks coefficients as alpha grows')
plt.axhline(0, color='black', linewidth=0.8)
plt.show()

## Reading the Ridge diagram

Moving right means a larger alpha. Coefficients move closer to zero, making the model less sensitive to noise. In contrast to Lasso, Ridge usually keeps every feature with a small non-zero coefficient.

In [ ]:
# Cell 4: fit a simple Ridge model with a chosen alpha
ridge_model = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
ridge_model.fit(X_train, y_train)
ridge_prediction = ridge_model.predict(X_test)
print('Ridge MAE:', round(mean_absolute_error(y_test, ridge_prediction), 3))
print('Ridge R2:', round(r2_score(y_test, ridge_prediction), 3))

### What is alpha?

Alpha controls the penalty. A tiny alpha behaves like ordinary Linear Regression. A very large alpha can underfit because it shrinks useful coefficients too much. Instead of guessing, RidgeCV can choose from several alpha values.

In [ ]:
# Cell 5: RidgeCV chooses alpha through 5-fold cross-validation
alpha_grid = np.logspace(-3, 3, 60)
ridge_cv_model = make_pipeline(StandardScaler(), RidgeCV(alphas=alpha_grid, cv=5, scoring='neg_mean_absolute_error'))
ridge_cv_model.fit(X_train, y_train)
ridge_cv_prediction = ridge_cv_model.predict(X_test)
chosen_ridge = ridge_cv_model.named_steps['ridgecv']
print('Best Ridge alpha:', chosen_ridge.alpha_)
print('RidgeCV MAE:', round(mean_absolute_error(y_test, ridge_cv_prediction), 3))
print('RidgeCV R2:', round(r2_score(y_test, ridge_cv_prediction), 3))

In [ ]:
# Cell 6: ElasticNet uses both L1 and L2 penalties
elastic_model = make_pipeline(StandardScaler(), ElasticNet(alpha=0.05, l1_ratio=0.5, max_iter=20000))
elastic_model.fit(X_train, y_train)
elastic_prediction = elastic_model.predict(X_test)
print('ElasticNet MAE:', round(mean_absolute_error(y_test, elastic_prediction), 3))
print('ElasticNet R2:', round(r2_score(y_test, elastic_prediction), 3))

### ElasticNet settings made easy

- **alpha** is the total amount of regularization.
- **l1_ratio = 1** means all Lasso (L1).
- **l1_ratio = 0** means all Ridge (L2).
- A value between 0 and 1 mixes both penalties.

The values above are a starting example; cross-validation is better for choosing them.

In [ ]:
# Cell 7: ElasticNetCV chooses alpha and the L1/L2 mix
elastic_cv_model = make_pipeline(
    StandardScaler(),
    ElasticNetCV(l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9], cv=5, max_iter=20000, random_state=42)
)
elastic_cv_model.fit(X_train, y_train)
elastic_cv_prediction = elastic_cv_model.predict(X_test)
chosen_elastic = elastic_cv_model.named_steps['elasticnetcv']
print('Best ElasticNet alpha:', chosen_elastic.alpha_)
print('Best L1 ratio:', chosen_elastic.l1_ratio_)
print('ElasticNetCV MAE:', round(mean_absolute_error(y_test, elastic_cv_prediction), 3))
print('ElasticNetCV R2:', round(r2_score(y_test, elastic_cv_prediction), 3))

In [ ]:
# Cell 8: compare all models on the same untouched test set
linear_model = make_pipeline(StandardScaler(), LinearRegression()).fit(X_train, y_train)
model_predictions = {
    'Linear Regression': linear_model.predict(X_test),
    'Ridge': ridge_prediction,
    'RidgeCV': ridge_cv_prediction,
    'ElasticNet': elastic_prediction,
    'ElasticNetCV': elastic_cv_prediction
}
comparison = pd.DataFrame([
    [name, mean_absolute_error(y_test, pred), r2_score(y_test, pred)]
    for name, pred in model_predictions.items()
], columns=['Model', 'Test MAE (lower is better)', 'Test R2 (higher is better)'])
comparison.sort_values('Test MAE (lower is better)')

In [ ]:
# Cell 9: visual comparison of actual and RidgeCV predictions
lower = min(y_test.min(), ridge_cv_prediction.min())
upper = max(y_test.max(), ridge_cv_prediction.max())
plt.figure(figsize=(6, 5))
plt.scatter(y_test, ridge_cv_prediction, alpha=0.8, color='#4C78A8')
plt.plot([lower, upper], [lower, upper], '--', color='crimson', label='Perfect prediction')
plt.xlabel('Actual FWI')
plt.ylabel('RidgeCV predicted FWI')
plt.title('RidgeCV final test-set check')
plt.legend()
plt.show()

## Quick revision card

1. Ridge is L2 regularization: it shrinks coefficients smoothly.
2. Lasso is L1 regularization: it can set coefficients to zero.
3. ElasticNet mixes L1 and L2 using l1_ratio.
4. RidgeCV and ElasticNetCV choose settings with cross-validation on training data.
5. Compare MAE and R² on the untouched test set; do not choose a model from one metric alone.
6. Only after evaluation, save the chosen pipeline for future predictions.

**One-line interview answer:** Use Ridge when related features should stay in the model; use ElasticNet when you want a balance between Ridge stability and Lasso-style feature selection.